# Hands-on 38 (Bab 15): Proyek Akhir End-to-End

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Capstone: simulator -> MQTT -> subscriber Python (validasi payload, preprocessing,
threshold baseline + Isolation Forest) -> storage -> alert. Struktur skrip mengikuti
buku: `scripts/publishers/`, `scripts/inference/`, `scripts/subscribers/`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Latih Model Baseline Isolation Forest

Baseline dilatih pada rentang operasi normal deployment (rentang payload normal
publisher); formula model persis buku. Lihat docstring
`scripts/inference/train_capstone_iforest.py` untuk alasannya.

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/inference/train_capstone_iforest.py"], capture_output=True, text=True)
print(r.stdout)

[SUKSES] Model disimpan ke: data/models/capstone_iforest.joblib



## 2. Jalankan Sistem (broker + subscriber+alert + listener + publisher)

In [3]:
import time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/subscribers/capstone_subscriber_with_alert.py",
         "--max-messages", "15", "--timeout", "45"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    listener = subprocess.Popen(
        [sys.executable, "scripts/mqtt/subscribe_sensor.py",
         "--topic", "capstone/iotnode01/alert", "--max-messages", "1", "--timeout", "45"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    subprocess.run([sys.executable, "scripts/publishers/capstone_publisher.py",
                    "--count", "15", "--interval", "0.3", "--seed", "7"],
                   capture_output=True, text=True)
    out_sub, _ = sub.communicate()
    print("--- Subscriber + alert ---")
    print(out_sub)
    out_listener, _ = listener.communicate()
    print("--- Listener topik alert ---")
    print(out_listener[:500])
finally:
    broker.terminate()

--- Subscriber + alert ---
[INFO] Connected to broker.
[INFO] Subscribed to capstone/iotnode01/telemetry
[2026-10-22T10:52:21] -> anomaly=0, threshold=0
[2026-10-22T10:52:22] -> anomaly=0, threshold=0
[2026-10-22T10:52:22] -> anomaly=0, threshold=0
[2026-10-22T10:52:22] -> anomaly=0, threshold=0
[2026-10-22T10:52:23] -> anomaly=1, threshold=0
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T10:52:23", "device_id": "iotnode01", "alert_reason": "isolation_forest", "temperature_c": 33.76, "humidity_pct": 41.63, "power_watt": 234.52, "current_amp": 5.24}
[2026-10-22T10:52:23] -> anomaly=0, threshold=0
[2026-10-22T10:52:23] -> anomaly=0, threshold=0
[2026-10-22T10:52:24] -> anomaly=0, threshold=0
[2026-10-22T10:52:24] -> anomaly=0, threshold=0
[2026-10-22T10:52:24] -> anomaly=0, threshold=0
[2026-10-22T10:52:24] -> anomaly=1, threshold=1
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T10:52:24", "device_id": "iotnode01", "alert_reason": "threshold+isolation_forest", "temperature_c": 28.18, "humi

## 3. Verifikasi Storage

In [4]:
import pandas as pd
mon = pd.read_csv("outputs/reports/capstone_monitoring.csv")
alerts = pd.read_csv("outputs/reports/capstone_alerts.csv")
print(mon.head(8).to_string(index=False))
print(f"\nAnomali: {mon['iforest_anomaly'].sum()}, Threshold: {mon['threshold_alert'].sum()}, Alert: {len(alerts)}")

          timestamp device_id  temperature_c  humidity_pct  power_watt  current_amp  iforest_anomaly  threshold_alert
2026-10-22T10:52:21 iotnode01          27.24         45.28  197.170000       3.9300                0                0
2026-10-22T10:52:22 iotnode01          27.66         42.03  171.340000       3.7200                0                0
2026-10-22T10:52:22 iotnode01          24.70         43.17  156.410000       8.4600                0                0
2026-10-22T10:52:22 iotnode01          26.23         61.96  250.590000       6.9600                0                0
2026-10-22T10:52:23 iotnode01          33.76         41.63  234.520000       5.2400                1                0
2026-10-22T10:52:23 iotnode01          25.18         50.80  226.900000       4.5800                0                0
2026-10-22T10:52:23 iotnode01          30.39         53.03  279.185871       7.9212                0                0
2026-10-22T10:52:24 iotnode01          28.28         51.

## Penutup

Sistem capstone berjalan ujung ke ujung dan alert sampai ke topik terpisah. Baseline
training memakai rentang operasi normal deployment (rentang payload normal publisher),
sesuai prinsip Bab 13 bahwa data training harus mewakili kondisi deployment - formula
model tetap persis buku (IsolationForest, n_estimators=200, contamination=0.05).
Hasilnya alert turun dari 14 menjadi 3 per 15 pesan: injeksi anomali tertangkap
(threshold + iforest), plus flag batas sesekali yang wajar untuk contamination 5% -
"notifikasi harus cukup berguna, bukan terlalu berisik" (Bab 15). Untuk deployment
penuh, aktifkan `--influx` dan susun dashboard Grafana (panel suhu, kelembaban, daya,
arus, iforest_anomaly, threshold_alert, histori, dan jumlah alert per jam).